In [2]:
"""
SentinelIQ — Dataset Fix Script
================================
Fixes all 5 critical/warning issues found in the audit:
  1. Drop risk_score leakage from feature vector
  2. Fix is_tor=True in normal rows
  3. Fix VPN ISPs in normal rows + hour distribution
  4. Add 6 missing features
  5. Add subtle near-border anomalies for realism
  6. Retrain + tune contamination → target Precision>75%, Recall>87%

Run:
    python fix_dataset.py

Output:
    nepal_auth_logins_fixed.csv   <- cleaned dataset
"""

import pandas as pd
import numpy as np
import joblib
import os

# Base directory = folder where this script lives
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
ARTIFACTS_DIR = os.path.join(BASE_DIR, "artifacts")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
import hashlib, random, warnings
from datetime import datetime, timedelta
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (classification_report, precision_score,
                             recall_score, f1_score, confusion_matrix)
import shap

warnings.filterwarnings("ignore")
np.random.seed(42)
random.seed(42)

# ─────────────────────────────────────────────
# 0. LOAD
# ─────────────────────────────────────────────
print("Loading dataset...")
INPUT_CSV = os.path.join(DATA_DIR, "nepal_auth_logins_merged.csv")
if not os.path.exists(INPUT_CSV):
    print(f"ERROR: Input file not found at {INPUT_CSV}")
    print("Place nepal_auth_logins_merged.csv inside the data/ folder")
    import sys; sys.exit(1)
df = pd.read_csv(INPUT_CSV)
print(f"  Original shape: {df.shape}")
print(f"  Label counts:\n{df['label'].value_counts()}\n")

# ─────────────────────────────────────────────
# FIX 1 — risk_score is label leakage
# ─────────────────────────────────────────────
print("FIX 1: Isolating risk_score as display-only column...")
df.rename(columns={"risk_score": "display_risk_score"}, inplace=True)
print("  risk_score -> display_risk_score (excluded from feature vector)\n")

# ─────────────────────────────────────────────
# FIX 2 — is_tor=True in 50 normal rows
# ─────────────────────────────────────────────
print("FIX 2: Removing Tor from normal rows...")
LEGIT_ISPS = ['NTC (Nepal Telecom)', 'Ncell (Mobile)', 'Vianet',
              'Subisu', 'WorldLink', 'Dishhome Fiber', 'UTL']

before = int(df[df['label'] == 'normal']['is_tor'].sum())
tor_normal_mask = (df['label'] == 'normal') & (df['is_tor'] == True)
df.loc[tor_normal_mask, 'is_tor'] = False
df.loc[tor_normal_mask, 'isp_name'] = np.random.choice(LEGIT_ISPS, size=tor_normal_mask.sum())
after = int(df[df['label'] == 'normal']['is_tor'].sum())
print(f"  is_tor in normal rows: {before} -> {after}\n")

# ─────────────────────────────────────────────
# FIX 3a — VPN ISPs in normal rows
# ─────────────────────────────────────────────
print("FIX 3a: Replacing VPN ISPs in normal rows...")
VPN_ISPS = ['NordVPN', 'ExpressVPN']
vpn_normal_mask = (df['label'] == 'normal') & (df['isp_name'].isin(VPN_ISPS))
n_vpn = int(vpn_normal_mask.sum())
df.loc[vpn_normal_mask, 'isp_name']     = np.random.choice(LEGIT_ISPS, size=n_vpn)
df.loc[vpn_normal_mask, 'is_known_vpn'] = False
df.loc[vpn_normal_mask, 'ip_type']      = 'residential'
print(f"  Replaced {n_vpn} VPN ISP entries with Nepali ISPs\n")

# ─────────────────────────────────────────────
# FIX 3b — Hour distribution
# ─────────────────────────────────────────────
print("FIX 3b: Fixing hour distribution in normal rows...")
normal_mask   = df['label'] == 'normal'
midnight_mask = normal_mask & (df['hour_of_day'] <= 5)
n_midnight    = int(midnight_mask.sum())

# Keep a realistic tail of late-night normal logins
keep_late = max(1, int(n_midnight * 0.10))
late_idx  = df[midnight_mask].sample(n=keep_late, random_state=42).index
df.loc[late_idx, 'hour_of_day'] = np.random.randint(0, 6, size=keep_late)

shift_idx = df[midnight_mask].index.difference(late_idx)
new_hours = np.clip(
    np.round(np.random.normal(14.0, 3.5, size=len(shift_idx))).astype(int), 6, 23
)
df.loc[shift_idx, 'hour_of_day'] = new_hours

after_mid  = int((df[normal_mask]['hour_of_day'] <= 5).sum())
after_pct  = after_mid / normal_mask.sum() * 100
print(f"  Midnight logins (normal): {n_midnight} -> {after_mid} ({after_pct:.1f}%)\n")

# ─────────────────────────────────────────────
# FIX 4 — Add 6 missing features
# ─────────────────────────────────────────────
print("FIX 4: Adding 6 missing features...")
n          = len(df)
normal_idx = df[df['label'] == 'normal'].index
anom_idx   = df[df['label'] == 'anomaly'].index

velocity_norm = df['velocity_last_1hr'].clip(0, 30) / 30.0
new_device_flag = df['is_new_device'].astype(int)

# ms_between_attempts
base_ms = np.random.lognormal(mean=9.3, sigma=0.6, size=n)
ms = base_ms * (1 - 0.7 * velocity_norm)
ms = np.clip(ms, 50, 60000)
df['ms_between_attempts'] = ms.round(1)
print("  + ms_between_attempts")

# fail_count_before_success
lam = 0.3 + 1.5 * velocity_norm + 0.8 * new_device_flag
fail = np.random.poisson(lam).clip(0, 20)
df['fail_count_before_success'] = fail
print("  + fail_count_before_success")

# device_trust_score
dts = np.where(
    df['is_new_device'],
    np.random.choice([0.5, 1.0], size=n, p=[0.7, 0.3]),
    0.0,
 )
df['device_trust_score'] = dts
print("  + device_trust_score")

# user_baseline_hour_mean — per-user consistent baseline
user_baselines = {uid: float(np.random.normal(13.0, 2.5))
                  for uid in df['user_id'].unique()}
df['user_baseline_hour_mean'] = df['user_id'].map(user_baselines).round(1)
df['hour_deviation'] = (df['hour_of_day'] - df['user_baseline_hour_mean']).abs().round(1)
print("  + user_baseline_hour_mean")
print("  + hour_deviation (recalculated)")

# user_account_age_days
base_age = np.random.lognormal(mean=5.4, sigma=0.7, size=n)
age = np.clip(base_age, 1, 2000)
age = np.where(df['is_new_device'], age * np.random.uniform(0.2, 0.6, size=n), age)
df['user_account_age_days'] = age.astype(int)
print("  + user_account_age_days")

# ip_reputation
network_risk = (
    0.05 +
    0.25 * df['is_known_vpn'].astype(int) +
    0.25 * df['is_datacenter_ip'].astype(int) +
    0.35 * df['is_tor'].astype(int) +
    0.15 * (df['velocity_last_24hr'].clip(0, 50) / 50.0)
 )
ip_rep = network_risk + np.random.normal(0.0, 0.05, size=n)
ip_rep = np.clip(ip_rep, 0.0, 0.95)
df['ip_reputation'] = ip_rep.round(3)
print("  + ip_reputation\n")

# ─────────────────────────────────────────────
# FIX 5 — Subtle near-border anomalies
# ─────────────────────────────────────────────
print("FIX 5: Adding 90 subtle near-border anomaly rows...")

def fake_hash(prefix):
    return prefix + hashlib.md5(str(random.random()).encode()).hexdigest()[:12]

border_profiles = [
    {"country": "IN", "city": "Raxaul",   "asn": "AS45609", "isp": "Airtel India",   "lat": 26.98, "lon": 84.85},
    {"country": "IN", "city": "Siliguri", "asn": "AS24560", "isp": "Bharti Airtel",  "lat": 26.73, "lon": 88.43},
    {"country": "BD", "city": "Dhaka",   "asn": "AS45609", "isp": "Grameenphone",  "lat": 23.81, "lon": 90.41},
    {"country": "BD", "city": "Khulna",  "asn": "AS24432", "isp": "Banglalink",    "lat": 22.85, "lon": 89.54},
    {"country": "IN", "city": "Patna",   "asn": "AS55836", "isp": "Reliance Jio",  "lat": 25.61, "lon": 85.13},
 ]

subtle = []
all_uids = df['user_id'].unique().tolist()
browsers = ["Chrome Mobile", "Samsung Internet", "Firefox", "Edge"]

for i in range(90):
    profile = random.choice(border_profiles)
    uid      = random.choice(all_uids)
    bl_hr    = user_baselines.get(uid, 13.0)
    hour     = int(np.clip(np.random.normal(13, 3.5), 5, 23))
    is_vpn   = bool(np.random.choice([True, False], p=[0.1, 0.9]))
    ip_type  = np.random.choice(["mobile", "residential"], p=[0.85, 0.15])

    subtle.append({
        "login_id":                    fake_hash("log_"),
        "user_id":                     uid,
        "timestamp":                   (datetime(2025,1,1) + timedelta(
                                            days=np.random.randint(0,180),
                                            hours=hour)).strftime("%Y-%m-%d %H:%M:%S"),
        "hour_of_day":                 hour,
        "day_of_week":                 np.random.randint(0,7),
        "ip_address":                  f"103.{np.random.randint(1,254)}.{np.random.randint(1,254)}.{np.random.randint(1,254)}",
        "asn":                         profile["asn"],
        "isp_name":                    profile["isp"],
        "ip_type":                     ip_type,
        "country":                     profile["country"],
        "city":                        profile["city"],
        "latitude":                    round(profile["lat"] + np.random.normal(0, 0.35), 4),
        "longitude":                   round(profile["lon"] + np.random.normal(0, 0.35), 4),
        "device_os":                   np.random.choice(["Android","iOS"], p=[0.8,0.2]),
        "device_browser":              random.choice(browsers),
        "device_fingerprint":          fake_hash("fp_"),
        "is_new_device":               bool(np.random.choice([True,False], p=[0.6,0.4])),
        "is_known_vpn":                is_vpn,
        "is_datacenter_ip":            False,
        "is_tor":                      False,
        "velocity_last_1hr":           int(np.random.randint(2, 10)),
        "velocity_last_24hr":          int(np.random.randint(4, 18)),
        "distance_from_last_login_km": round(float(np.random.uniform(150, 900)), 1),
        "hours_since_last_login":      round(float(np.random.uniform(0.4, 8.0)), 2),
        "impossible_travel":           False,
        "hour_deviation":              round(abs(hour - bl_hr), 1),
        "display_risk_score":          round(float(np.random.uniform(0.35, 0.75)), 3),
        "label":                       "anomaly",
        "anomaly_type":                "near_border_foreign_ip",
        "ms_between_attempts":         round(float(np.random.uniform(2000, 20000)), 1),
        "fail_count_before_success":   int(np.random.choice([0,1,2,3], p=[0.45,0.30,0.20,0.05])),
        "device_trust_score":          float(np.random.choice([0.0, 0.5, 1.0], p=[0.45,0.4,0.15])),
        "user_baseline_hour_mean":     round(bl_hr, 1),
        "user_account_age_days":       int(np.random.randint(5, 600)),
        "ip_reputation":               round(float(np.random.uniform(0.15, 0.7)), 3),
    })

df_subtle = pd.DataFrame(subtle)
df        = pd.concat([df, df_subtle], ignore_index=True)
df        = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"  Added 90 near-border rows | New shape: {df.shape}")
print(f"  Anomaly types:\n{df['anomaly_type'].value_counts()}\n")

# ─────────────────────────────────────────────
# SAVE FIXED DATASET
# ─────────────────────────────────────────────
out_csv = os.path.join(DATA_DIR, "nepal_auth_logins_fixed.csv")
df.to_csv(out_csv, index=False)
print(f"Fixed dataset saved -> {out_csv}\n")



Loading dataset...
  Original shape: (19882, 29)
  Label counts:
label
normal     18279
anomaly     1603
Name: count, dtype: int64

FIX 1: Isolating risk_score as display-only column...
  risk_score -> display_risk_score (excluded from feature vector)

FIX 2: Removing Tor from normal rows...
  is_tor in normal rows: 50 -> 0

FIX 3a: Replacing VPN ISPs in normal rows...
  Replaced 326 VPN ISP entries with Nepali ISPs

FIX 3b: Fixing hour distribution in normal rows...
  Midnight logins (normal): 2652 -> 265 (1.4%)

FIX 4: Adding 6 missing features...
  + ms_between_attempts
  + fail_count_before_success
  + device_trust_score
  + user_baseline_hour_mean
  + hour_deviation (recalculated)
  + user_account_age_days
  + ip_reputation

FIX 5: Adding 90 subtle near-border anomaly rows...
  Added 90 near-border rows | New shape: (19972, 35)
  Anomaly types:
anomaly_type
none                              18279
odd_hours_vpn                       420
impossible_travel                   411
crede